## Clone git, import data and unzip

In [1]:
!git clone https://github.com/890tsugua/AppliedML2026.git # Write in terminal

Cloning into 'AppliedML2026'...
remote: Enumerating objects: 313, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 313 (delta 44), reused 47 (delta 21), pack-reused 229 (from 1)
Receiving objects: 100% (313/313), 79.44 MiB | 15.62 MiB/s, done.
Resolving deltas: 100% (167/167), done.
Updating files: 100% (29/29), done.


In [2]:
import sys
import shutil
from pathlib import Path
import torch
#torch.backends.cudnn.benchmark = True

In [3]:
# Mount to the folder in google drive
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

data_dir = Path("/content/drive/MyDrive/AppliedMLdata2026")

print(data_dir.exists())
print(list(data_dir.iterdir())[:5])

Mounted at /content/drive
True
[PosixPath('/content/drive/MyDrive/AppliedMLdata2026/train_data'), PosixPath('/content/drive/MyDrive/AppliedMLdata2026/test_data'), PosixPath('/content/drive/MyDrive/AppliedMLdata2026/AppliedMLdata2026.zip'), PosixPath('/content/drive/MyDrive/AppliedMLdata2026/AppML_augmented_data.zip')]


In [4]:

sys.path.append("/content/AppliedML2026/country_cnn")
from src.dataset import make_dataloaders_from_dir, show_image

In [7]:
#drive_zip = Path("/content/drive/MyDrive/AppliedMLdata2026/AppliedMLdata2026.zip")
drive_zip = Path("/content/drive/MyDrive/AppliedMLdata2026/AppML_augmented_data.zip")
local_zip = Path("/content/zippedAppliedMLdata2026.zip")

shutil.copy2(drive_zip, local_zip)

PosixPath('/content/zippedAppliedMLdata2026.zip')

In [8]:
# Unzip locally
import zipfile

local_data = Path("/content/zippedAppliedMLdata2026")

with zipfile.ZipFile(local_zip, "r") as zf:
    zf.extractall(local_data)

In [9]:
from src.train import train
from src.model import make_model

## Prep done, make dataloader and model

In [37]:
train_loader, val_loader = make_dataloaders_from_dir("/content/zippedAppliedMLdata2026/AppML_augmented_data/train_data",
                                                     batch_size=128,
                                                     image_size=224,
                                                     num_workers=1,
                                                     prefetch_factor=2,
                                                     random_crop=False,
                                                     color_jitter=False,
                                                     rotation=False,
                                                     horizontal_flip=False
                                                     )

In [38]:
resnet50 = make_model(model_name="resnet50", trainable_layers=4)

In [13]:
%cd /content/AppliedML2026

/content/AppliedML2026


In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_name = "resnet50testgpua100"
resnet50 = resnet50.to(device)

history = train(resnet50, train_loader, val_loader, device, save_name, save_checkpoints=True, num_epochs=50)

Description:  20%|█▉        | 75/383 [00:40<02:47,  1.84it/s]


KeyboardInterrupt: 

## Testing GPU and CPU

In [14]:
import time

In [32]:
# Test loading time for one batch
start = time.time()
images, labels = next(iter(train_loader))
print(time.time() - start)

0.9208498001098633


In [ ]:
# Test GPU average speed for 1 preloaded batch.
# warmup
for _ in range(10):
    with torch.no_grad():
        _ = model(images)

torch.cuda.synchronize()

import time
start = time.time()

for _ in range(100):
    with torch.no_grad():
        _ = model(images)

torch.cuda.synchronize()

print((time.time() - start) / 100)

In [16]:
# Testing jpg decoding time
import time
from PIL import Image

paths = [p for p, _ in train_loader.dataset.dataset.samples[:1000]]

start = time.time()
for p in paths:
    img = Image.open(p).convert("RGB")
print("JPEG open+decode:", time.time() - start)

JPEG open+decode: 1.1158080101013184


## Optuna

In [13]:
num_classes=85
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 3e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)

    model = make_model(
        model_name="resnet34",
        num_classes=num_classes,
        pretrained=True
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    criterion = torch.nn.CrossEntropyLoss()

    history = train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        save_name=f"optuna_trial_{trial.number}",
        save_checkpoints=True,
        optimizer=optimizer,
        criterion=criterion,
        num_epochs=5
    )

    best_val_loss = min(history["val_loss"])
    return best_val_loss

In [14]:
import optuna

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print(study.best_params)
print(study.best_value)

[I 2026-05-27 12:58:34,606] A new study created in memory with name: no-name-8f13ff46-092e-446a-aead-9d94aa9e97b5
/content/AppliedML2026/country_cnn/src/train.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # For mixed precision training
Description:   0%|          | 0/255 [00:00<?, ?it/s]/content/AppliedML2026/country_cnn/src/train.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): # For mixed precision training
Description: 100%|██████████| 64/64 [00:04<00:00, 13.76it/s]


Saved best model
Epoch 1/5 | Train loss: 3.1055, acc: 0.2295, top5: 0.4988 | Val loss: 5.0663, acc: 0.0716, top5: 0.2845


Description: 100%|██████████| 64/64 [00:04<00:00, 14.89it/s]


Saved best model
Epoch 2/5 | Train loss: 2.2463, acc: 0.3869, top5: 0.7064 | Val loss: 4.6791, acc: 0.1045, top5: 0.2872


Description: 100%|██████████| 64/64 [00:04<00:00, 13.38it/s]


...

Epoch 2/5 | Train loss: 2.0178, acc: 0.4616, top5: 0.7616 | Val loss: 5.1770, acc: 0.0839, top5: 0.2746


Description: 100%|██████████| 64/64 [00:04<00:00, 13.30it/s]


Epoch 3/5 | Train loss: 1.5492, acc: 0.5711, top5: 0.8511 | Val loss: 4.8832, acc: 0.0996, top5: 0.3146


Description:  63%|██████▎   | 161/255 [00:29<00:17,  5.42it/s]
[W 2026-05-27 14:00:35,717] Trial 14 failed with parameters: {'lr': 0.0003399072437554149, 'weight_decay': 1.2461519059298028e-06} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_2456/4231168982.py", line 21, in objective
    history = train(
              ^^^^^^
  File "/content/AppliedML2026/country_cnn/src/train.py", line 96, in train
    train_loss, train_accuracy, train_accuracy_top5 = run_epoch(model, train_loader, optimizer, criterion, device, train=True)
                                                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/AppliedML2026/country_cnn/src/train.py", line 25, in run_epoch
    for batch_idx, (images, labe

KeyboardInterrupt: 

In [15]:
print(study.best_params)
print(study.best_value)

{'lr': 0.001916445324514403, 'weight_decay': 9.256459432630416e-06}
3.7031984876452153


In [16]:
# Write to a text file in outputs
from pathlib import Path

# Create output directory if it does not exist
output_dir = Path("content/AppliedML2026/country_cnn/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# File path
best_params_path = output_dir / "best_params.txt"

# Write Optuna results
with open(best_params_path, "w") as f:
    f.write(f"Best value: {study.best_value}\n\n")
    f.write("Best parameters:\n")

    for key, value in study.best_params.items():
        f.write(f"{key}: {value}\n")

print(f"Saved best params to {best_params_path}")

Saved best params to content/AppliedML2026/country_cnn/outputs/best_params.txt
